In [6]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

import json
import datetime

import pandas as pd
#from langchain.schema import HumanMessage

In [ ]:
def test_llm(LLMS, gameSetupFile):
    
    with open(gameSetupFile, "r") as f:
        data = json.load(f)["games"]
        
    results = []
    
    for model in LLMS:
        print("===", model, "===")
        print(35*"=")
        llm = ChatOllama(
            model=model,
            temperature=0.1,
            base_url="http://10.203.222.234:11434/",
            keep_alive=-1
        )
        
    for question in data:
            try:
                response = llm.invoke([
                    SystemMessage(content=question["instructions"]),
                    HumanMessage(content=question["board"])
                ])
                answer = response.content.strip()

            except Exception as e:
                print(f"{model} not available — {e}", end=": ")
                answer = None
                break

            results.append({
                "model": model,
                "question": question,
                "answer": answer
            })
            print("=", end="")
            
    return results

In [ ]:
LLMS = [
    # "everythinglm:13b",
    'gpt-oss:20b', 
    'gemma3:27b', 
    'gpt-oss:120b',
    "deepseek-r1:8b",
    'deepseek-r1:70b',
    ]
questions_file = "popCulture.json"

timestamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

df_results = test_llm(LLMS, questions_file)
df_results.to_csv(f"results/results_{timestamp}.csv")
print(df_results)